# Data Analyst Assignment 2 — Solución Completa

**Objetivo:** Analizar cómo el entrenamiento de empleados impacta el desempeño de ventas.

**Tablas disponibles:**
- `Employee_Data` — Información de empleados (2,532 filas)
- `Completed_Trainings` — Cursos completados por empleado (1,982 filas)
- `Performance Data` — Oportunidades de ventas (607 filas)

## 0. Importar librerías y cargar datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

xl = pd.read_excel('Assignment_2.xlsx', sheet_name=None)
emp   = xl['Employee_Data']
train = xl['Completed_Trainings']
perf  = xl['Performance Data']

print('Employee_Data shape:', emp.shape)
print('Completed_Trainings shape:', train.shape)
print('Performance Data shape:', perf.shape)

In [ ]:

print('=== Employee_Data ===')
display(emp.head(3))
print('=== Completed_Trainings ===')
display(train.head(3))
print('=== Performance Data ===')
display(perf.head(3))

---
## Pregunta 1 — Tasa de Completitud de Entrenamiento por SVP Leader y Región

### Reglas de negocio aplicadas:
| Condición | Regla |
|-----------|-------|
| Empleados en licencia (`On Leave`) | **Exentos** de todos los cursos |
| `Sell More Suite SKU` | **No requerido** para `Advocacy` |
| `Suite/Automation Technical Lab` y `Advanced Suite Bots Lab Course` | **Solo requeridos** para `PreSales` y `Services` |

In [ ]:

active_emp = emp[emp['Leave Status'] == 'Active'].copy()
print(f"Empleados activos (sujetos a entrenamiento): {len(active_emp):,}")
print(f"Empleados exentos (On Leave): {len(emp) - len(active_emp):,}")

In [ ]:

ALL_COURSES = [
    'Sell More Suite SKU',
    'Suite/Automation Technical Lab',
    'Advanced Suite Bots Lab Course'
]

def is_required(cost_center_family, course):
    """Retorna True si el curso es requerido para la familia de cost center."""
    if course == 'Sell More Suite SKU':
        return cost_center_family != 'Advocacy'
    elif course in ['Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']:
        return cost_center_family in ['PreSales', 'Services']
    return True

records = []
for _, row in active_emp.iterrows():
    for course in ALL_COURSES:
        if is_required(row['Cost Center Family'], course):
            records.append({
                'Employee_ID': row['Employee_ID'],
                'SVP Leader': row['SVP Leader'],
                'Region': row['Region'],
                'Cost Center Family': row['Cost Center Family'],
                'Course': course
            })

required_df = pd.DataFrame(records)
print(f"Pares (empleado × curso) requeridos: {len(required_df):,}")

In [ ]:

train_clean = train.dropna(subset=['Employee_ID']).copy()
train_clean['Employee_ID'] = train_clean['Employee_ID'].astype(int)

completed_set = set(
    zip(train_clean['Employee_ID'], train_clean['Training_Completed'])
)

required_df['Completed'] = required_df.apply(
    lambda r: 1 if (r['Employee_ID'], r['Course']) in completed_set else 0,
    axis=1
)

print('Completitud general por curso:')
display(
    required_df.groupby('Course')['Completed']
    .agg(Required='count', Completed='sum')
    .assign(**{'Completion Rate': lambda d: (d['Completed']/d['Required']).map('{:.1%}'.format)})
)

In [ ]:

by_svp = (
    required_df
    .groupby(['SVP Leader', 'Course'])['Completed']
    .agg(Required='count', Completed='sum')
    .reset_index()
)
by_svp['Completion Rate'] = by_svp['Completed'] / by_svp['Required']

svp_pivot = by_svp.pivot_table(
    index='SVP Leader',
    columns='Course',
    values='Completion Rate',
    aggfunc='mean'
).round(4)

print('Tasa de Completitud por SVP Leader:')
display(svp_pivot.style.format('{:.1%}').background_gradient(cmap='RdYlGn', axis=None))

In [ ]:

by_region = (
    required_df
    .groupby(['Region', 'Course'])['Completed']
    .agg(Required='count', Completed='sum')
    .reset_index()
)
by_region['Completion Rate'] = by_region['Completed'] / by_region['Required']

region_pivot = by_region.pivot_table(
    index='Region',
    columns='Course',
    values='Completion Rate'
).round(4)

print('Tasa de Completitud por Región:')
display(region_pivot.style.format('{:.1%}').background_gradient(cmap='RdYlGn', axis=None))

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
region_plot = by_region.copy()
region_plot['Course_Short'] = region_plot['Course'].str.replace('Suite/Automation Technical Lab', 'Tech Lab').str.replace('Advanced Suite Bots Lab Course', 'Bots Lab').str.replace('Sell More Suite SKU', 'Sell More SKU')
for i, course in enumerate(region_plot['Course_Short'].unique()):
    subset = region_plot[region_plot['Course_Short'] == course]
    x = range(len(subset))
    ax.bar([xi + i*0.25 for xi in x], subset['Completion Rate'], width=0.25, label=course)
ax.set_xticks([xi + 0.25 for xi in range(4)])
ax.set_xticklabels(['APAC', 'EMEA', 'LATAM', 'North America'])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Completion Rate by Region & Course', fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.1)
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='80% target')
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
sell_svp = by_svp[by_svp['Course'] == 'Sell More Suite SKU'].sort_values('Completion Rate')
colors = ['#d73027' if r < 0.5 else '#fee090' if r < 0.75 else '#1a9850' for r in sell_svp['Completion Rate']]
ax2.barh(sell_svp['SVP Leader'], sell_svp['Completion Rate'], color=colors)
ax2.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_title('Sell More SKU: Completion Rate by SVP Leader', fontweight='bold')
ax2.axvline(0.5, color='red', linestyle='--', alpha=0.7)
ax2.set_xlim(0, 1.05)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('q1_completion_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado.')

### Hallazgos Pregunta 1

- **`Sell More Suite SKU`** tiene la tasa de completitud más baja (~41%). Es el curso requerido para el mayor número de empleados (1,941 personas).
- **`Suite/Automation Technical Lab`** y **`Advanced Suite Bots Lab Course`** tienen ~90% de completitud, pero aplican solo a PreSales y Services (~429 empleados).
- Por región, **APAC** tiene la tasa más baja en `Sell More SKU` (7.7%), mientras que **North America** lidera (53.9%).
- **Leader 13** tiene 0% en `Sell More SKU` — requiere atención inmediata.

---
## Pregunta 2 — Desempeño de Account Executives: Entrenados vs. No Entrenados

**Enfoque:** Comparar a todos los IC (Individual Contributors) que completaron al menos UN entrenamiento vs. los que no completaron ninguno, sin distinción de curso específico.

In [ ]:

ic_emp = emp[emp['Manager IC Helper'] == 'IC'].copy()
print(f"Total ICs: {len(ic_emp):,}")

trained_ids = set(train_clean['Employee_ID'].unique())
ic_emp['Trained'] = ic_emp['Employee_ID'].isin(trained_ids)

print(f"ICs con entrenamiento: {ic_emp['Trained'].sum():,}")
print(f"ICs sin entrenamiento: {(~ic_emp['Trained']).sum():,}")

In [ ]:

perf_clean = perf.dropna(subset=['Employee_ID']).copy()
perf_clean['Employee_ID'] = perf_clean['Employee_ID'].astype(int)

perf_ic = perf_clean.merge(
    ic_emp[['Employee_ID', 'Trained', 'SVP Leader', 'Region', 'Cost Center Family']],
    on='Employee_ID', how='inner'
)

print(f"Oportunidades de ICs con datos: {len(perf_ic):,}")

perf_ic['Is_Closed_Won'] = perf_ic['Stage'] == '06 - Signed'

In [ ]:

summary = (
    perf_ic.groupby('Trained')
    .agg(
        Unique_AEs       = ('Employee_ID', 'nunique'),
        Total_Opps       = ('Opportunity ID', 'count'),
        Closed_Won       = ('Is_Closed_Won', 'sum'),
        Total_ARR        = ('Total Commissionable ARR (converted)', 'sum'),
        Avg_ARR_per_deal = ('Total Commissionable ARR (converted)', 'mean'),
    )
    .reset_index()
)
summary['Win_Rate']        = summary['Closed_Won'] / summary['Total_Opps']
summary['ARR_per_AE']      = summary['Total_ARR'] / summary['Unique_AEs']
summary['Opps_per_AE']     = summary['Total_Opps'] / summary['Unique_AEs']
summary['Trained_Label']   = summary['Trained'].map({True: 'Trained', False: 'Not Trained'})

display_cols = ['Trained_Label','Unique_AEs','Total_Opps','Opps_per_AE',
                'Closed_Won','Win_Rate','Total_ARR','Avg_ARR_per_deal','ARR_per_AE']
print('Resumen Global — Entrenados vs No Entrenados:')
display(
    summary[display_cols].style
    .format({'Win_Rate': '{:.1%}', 'Total_ARR': '${:,.0f}',
             'Avg_ARR_per_deal': '${:,.0f}', 'ARR_per_AE': '${:,.0f}',
             'Opps_per_AE': '{:.1f}'})
    .set_caption('AE Performance: Trained vs Not Trained')
)

In [ ]:

by_region_perf = (
    perf_ic.groupby(['Region', 'Trained'])
    .agg(
        Unique_AEs   = ('Employee_ID', 'nunique'),
        Total_Opps   = ('Opportunity ID', 'count'),
        Closed_Won   = ('Is_Closed_Won', 'sum'),
        Total_ARR    = ('Total Commissionable ARR (converted)', 'sum'),
    )
    .reset_index()
)
by_region_perf['Win_Rate']   = by_region_perf['Closed_Won'] / by_region_perf['Total_Opps']
by_region_perf['ARR_per_AE'] = by_region_perf['Total_ARR'] / by_region_perf['Unique_AEs']
by_region_perf['Trained_Label'] = by_region_perf['Trained'].map({True: 'Trained', False: 'Not Trained'})

print('Desempeño por Región:')
display(
    by_region_perf[['Region','Trained_Label','Unique_AEs','Total_Opps','Closed_Won','Win_Rate','ARR_per_AE']]
    .style.format({'Win_Rate': '{:.1%}', 'ARR_per_AE': '${:,.0f}'})
)

In [ ]:

by_svp_perf = (
    perf_ic.groupby(['SVP Leader', 'Trained'])
    .agg(
        Unique_AEs = ('Employee_ID', 'nunique'),
        Total_Opps = ('Opportunity ID', 'count'),
        Closed_Won = ('Is_Closed_Won', 'sum'),
        Total_ARR  = ('Total Commissionable ARR (converted)', 'sum'),
    )
    .reset_index()
)
by_svp_perf['Win_Rate']    = by_svp_perf['Closed_Won'] / by_svp_perf['Total_Opps']
by_svp_perf['ARR_per_AE']  = by_svp_perf['Total_ARR'] / by_svp_perf['Unique_AEs']
by_svp_perf['Trained_Label'] = by_svp_perf['Trained'].map({True: 'Trained', False: 'Not Trained'})
print('Desempeño por SVP Leader:')
display(
    by_svp_perf[['SVP Leader','Trained_Label','Unique_AEs','Total_Opps','Win_Rate','ARR_per_AE']]
    .sort_values(['SVP Leader','Trained'])
    .style.format({'Win_Rate': '{:.1%}', 'ARR_per_AE': '${:,.0f}'})
)

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_map = {True: '#2196F3', False: '#FF7043'}
labels_map = {True: 'Trained', False: 'Not Trained'}

metrics = [
    ('Win_Rate',    'Win Rate',          '% of deals closed'),
    ('ARR_per_AE',  'ARR per AE (USD)',  'Annual Recurring Revenue per AE'),
    ('Opps_per_AE', 'Opps per AE',       'Pipeline Opportunities per AE'),
]

for ax, (col, title, ylabel) in zip(axes[:2], metrics[:2]):
    vals = summary[col].values
    clrs = [colors_map[t] for t in summary['Trained'].values]
    lbls = [labels_map[t] for t in summary['Trained'].values]
    bars = ax.bar(lbls, vals, color=clrs, edgecolor='white', linewidth=1.5)
    if col == 'Win_Rate':
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')
    else:
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000, f'${v/1e3:.0f}K', ha='center', va='bottom', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)

ax3 = axes[2]
regions = by_region_perf['Region'].unique()
x = np.arange(len(regions))
width = 0.35
for i, trained_val in enumerate([False, True]):
    sub = by_region_perf[by_region_perf['Trained'] == trained_val]
    sub = sub.set_index('Region').reindex(regions)
    ax3.bar(x + i*width, sub['ARR_per_AE'].fillna(0), width,
            label=labels_map[trained_val], color=colors_map[trained_val])
ax3.set_xticks(x + width/2)
ax3.set_xticklabels(regions, rotation=15)
ax3.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax3.set_title('ARR per AE by Region', fontweight='bold')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

plt.suptitle('AE Performance: Trained vs Not Trained', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q2_performance.png', dpi=150, bbox_inches='tight')
plt.show()

### Hallazgos Pregunta 2

| Métrica | No Entrenados | Entrenados | Δ |
|---------|:---:|:---:|:---:|
| Win Rate | 13.7% | 13.6% | ~0% |
| ARR per AE | $156,657 | $236,075 | **+$79K (+51%)** |
| Avg ARR per deal | $78,329 | $107,243 | **+$29K (+37%)** |

**Conclusión:** La tasa de cierre (win rate) es similar entre grupos, pero los AEs entrenados generan **51% más ARR por persona** y cierran deals con tickets **37% mayores en promedio**. El entrenamiento no aumenta la cantidad de deals cerrados, sino la **calidad y valor** de cada uno.

**Por región:** EMEA y North America muestran el mayor uplift de entrenamiento en ARR per AE. APAC tiene un comportamiento atípico (pocos entrenados, distorsión de muestra pequeña).

---
## Pregunta 3 — Otros Insights, Calidad de Datos y Datos Adicionales Sugeridos

In [ ]:

print('=== CALIDAD DE DATOS ===')

print(f'\n1. Employee_IDs nulos en Completed_Trainings: {train["Employee_ID"].isna().sum()}')
print('   → 200 registros sin Employee_ID no pueden vincularse a ningún empleado.')

print(f'\n2. Employee_IDs nulos en Performance Data: {perf["Employee_ID"].isna().sum()}')

stages_present = perf['Stage'].unique().tolist()
print(f'\n3. Stages presentes en datos: {stages_present}')
print('   → El diccionario menciona "07 - Closed" como etapa final, pero NO existe en los datos.')
print('   → Solo existe "06 - Signed". Puede causar subconteo de deals cerrados.')

dup_opps = perf.groupby('Opportunity ID')['Employee_ID'].nunique()
multi_ae = dup_opps[dup_opps > 1]
print(f'\n4. Oportunidades asignadas a >1 empleado: {len(multi_ae)}')
if len(multi_ae) > 0:
    print('   → Puede generar double-counting de ARR si no se controla.')

perf_ids = set(perf_clean['Employee_ID'])
emp_ids  = set(emp['Employee_ID'])
print(f'\n5. Empleados en Performance no encontrados en Employee_Data: {len(perf_ids - emp_ids)}')

train_ids_int = set(train_clean['Employee_ID'])
print(f'6. Empleados en Training no encontrados en Employee_Data: {len(train_ids_int - emp_ids)}')

In [ ]:

print('=== OTROS INSIGHTS ===')

print('\n1. Mix de oportunidades por Tipo:')
display(perf_ic.groupby('Type').agg(
    Count=('Opportunity ID','count'),
    Total_ARR=('Total Commissionable ARR (converted)','sum'),
    Win_Rate=('Is_Closed_Won','mean')
).assign(**{'Win_Rate': lambda d: d['Win_Rate'].map('{:.1%}'.format),
            'Total_ARR': lambda d: d['Total_ARR'].map('${:,.0f}'.format)}))

print('\n2. % de ICs entrenados por Cost Center Family:')
ccf_train = ic_emp.groupby('Cost Center Family')['Trained'].mean().sort_values(ascending=False)
display(ccf_train.map('{:.1%}'.format).to_frame('% Trained'))

print('\n3. Distribución de empleados en licencia por Región:')
display(emp.groupby('Region')['Leave Status'].value_counts(normalize=True).map('{:.1%}'.format).unstack())

In [ ]:

additional_data = {
    'Dato Sugerido': [
        'Fecha de completitud del entrenamiento',
        'Fecha de ingreso / hire date del empleado',
        'Score / resultado del entrenamiento',
        'Cuota objetivo (quota) por AE',
        'Datos históricos de performance (multi-período)',
        'Oportunidades perdidas (Closed Lost)',
        'Número de actividades (calls, demos, emails)',
        'Información del cliente / account (tamaño, industria)',
        'Manager directo del AE (no solo SVP)',
        'Producto vendido en relación al entrenamiento específico',
    ],
    'Por qué es útil': [
        'Permitiría análisis de causalidad temporal: ¿mejoró performance DESPUÉS del entrenamiento?',
        'Controlar por seniority/tenure en análisis de performance',
        'Diferenciar a quienes completaron vs. completaron bien el entrenamiento',
        'Calcular % de cuota alcanzado para una métrica de performance más justa',
        'Identificar tendencias y estacionalidad, no solo snapshot actual',
        'Calcular win rate real; hoy solo hay una etapa de cierre en los datos',
        'Medir productividad de actividades, no solo resultado final',
        'Segmentar por complejidad de deal y controlar variables externas',
        'Analizar efecto del coaching del manager directo en adoption del entrenamiento',
        'Conectar el entrenamiento específico con el producto que el AE vende',
    ]
}
print('Datos Adicionales Sugeridos:')
display(pd.DataFrame(additional_data))

---
## Resumen Ejecutivo

### Q1 — Completitud de Entrenamiento
- **`Sell More Suite SKU`** (el de mayor alcance: 1,941 empleados requeridos) tiene solo un **41% de completitud** — la brecha más crítica.
- Los cursos técnicos (`Tech Lab`, `Bots Lab`) están casi al **90%**, pero solo aplican a ~429 personas (PreSales/Services).
- **Leader 13** tiene **0%** en Sell More SKU. **APAC** tiene solo **7.7%** — ambos son prioridades urgentes.

### Q2 — Impacto del Entrenamiento en Performance
- Los AEs entrenados generan **$236K ARR por persona vs. $157K** de los no entrenados (+51%).
- El ticket promedio por deal es **37% mayor** para entrenados.
- La tasa de cierre (win rate) es similar (~13.7%), lo que sugiere que el entrenamiento mejora la **calidad del deal** más que la velocidad de cierre.
- **Recomendación:** Priorizar completitud de entrenamiento en ICs activos que aún no han sido entrenados, especialmente en APAC y LATAM.

### Q3 — Calidad de Datos y Desafíos
- **200 registros de entrenamiento** sin Employee_ID válido — pérdida de datos de completitud.
- **Stage '07 - Closed'** documentado en el diccionario pero **ausente en los datos** — posible subconteo de cierres.
- La muestra de performance es pequeña (601 oportunidades para 285 ICs), lo que limita la significancia estadística por subgrupos.
- **Dato más valioso faltante:** fecha de completitud del entrenamiento para habilitar análisis de causalidad temporal.